# 14 — Swing Options

Swing options allow the holder to exercise multiple times during the option life,
subject to constraints (e.g., minimum rest between exercises).

- **FD pricing** via `fd_swing_price`
- Effect of exercise count on price
- JIT compilation speedup (often 10,000×+)
- AD Greeks: dV/dSpot, dV/dVol

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np

In [ ]:
from ql_jax.engines.fd.swing import fd_swing_price

S0, K = 100.0, 100.0
T, r, q, sigma = 1.0, 0.05, 0.02, 0.30

---
## 1. Single Exercise (= American Call)

In [ ]:
p1 = float(fd_swing_price(S0, K, T, r, q, sigma, n_exercises=1, n_x=200, n_t=200))
print(f"Swing with 1 exercise (≈ American call): {p1:.6f}")

---
## 2. Multiple Exercises

In [ ]:
n_ex = [1, 2, 3, 5, 10, 20]
prices = []

for n in n_ex:
    p = float(fd_swing_price(S0, K, T, r, q, sigma, n_exercises=n, n_x=200, n_t=200))
    prices.append(p)
    print(f"  n_exercises={n:3d}  price = {p:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(n_ex, prices, 'bo-', markersize=8)
plt.xlabel('Number of Exercises')
plt.ylabel('Swing Price')
plt.title('Swing Option: Price vs Exercise Count')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPrice ratio (20 ex / 1 ex) = {prices[-1] / prices[0]:.2f}×")

---
## 3. JIT Speedup

In [ ]:
# Non-JIT timing
nojit_t, _ = timed_ms(fd_swing_price, S0, K, T, r, q, sigma, 5, n_x=100, n_t=100)

# JIT version
jit_swing = jax.jit(fd_swing_price, static_argnames=['n_exercises', 'n_x', 'n_t'])
_ = jit_swing(S0, K, T, r, q, sigma, 5, n_x=100, n_t=100)  # warmup
jit_t, _ = timed_ms(jit_swing, S0, K, T, r, q, sigma, 5, n_x=100, n_t=100)

print(f"No JIT  : {nojit_t:8.1f} ms")
print(f"JIT     : {jit_t:8.1f} ms")
print(f"Speedup : {nojit_t / jit_t:,.0f}×")

---
## 4. AD Greeks

In [ ]:
def swing_price_for_ad(spot, vol):
    return fd_swing_price(spot, K, T, r, q, vol, n_exercises=5, n_x=100, n_t=100)

delta_fn = jax.grad(swing_price_for_ad, argnums=0)
vega_fn  = jax.grad(swing_price_for_ad, argnums=1)

delta = float(delta_fn(S0, sigma))
vega  = float(vega_fn(S0, sigma))

print(f"Swing delta (5 exercises) : {delta:.6f}")
print(f"Swing vega                : {vega:.6f}")

In [ ]:
# Delta vs spot
spots = jnp.linspace(70, 130, 50)
deltas = [float(delta_fn(s, sigma)) for s in spots]

plt.figure(figsize=(8, 5))
plt.plot(np.array(spots), deltas, 'b-', linewidth=2)
plt.xlabel('Spot')
plt.ylabel('Delta')
plt.title('Swing Option Delta (5 exercises, via AD)')
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. Grid Sensitivity

In [ ]:
grids = [50, 100, 200, 400]
grid_prices = []

for g in grids:
    p = float(fd_swing_price(S0, K, T, r, q, sigma, n_exercises=5, n_x=g, n_t=g))
    grid_prices.append(p)
    print(f"  grid {g:3d}×{g:3d} : {p:.6f}")

plt.figure(figsize=(8, 5))
plt.plot(grids, grid_prices, 'rs-', markersize=8)
plt.xlabel('Grid Size (n_x = n_t)')
plt.ylabel('Price')
plt.title('FD Swing: Convergence with Grid Refinement')
plt.grid(True, alpha=0.3)
plt.show()

---
## 6. Exercises

1. **Rest constraint**: Vary `min_rest` from 1 to 10 and observe its effect on the swing price.
2. **vmap over strikes**: Batch-price swing options across 100 strikes using `jax.vmap`.
3. **Gamma**: Compute $\partial^2 V / \partial S^2$ via `jax.grad(jax.grad(...))` and plot it.

---
*End of Notebook 14*